In [1]:
import cloudViewer as cv3d
import numpy as np
import matplotlib.pyplot as plt
import os
import sys

# only needed for tutorial, monkey patches visualization
sys.path.append('..')
import cloudViewer_tutorial as cv3dtut
# change to True if you want to interact with the visualization windows
cv3dtut.interactive = not "CI" in os.environ

Using external CloudViewer-ML in /root/CloudViewer-ML


# Surface reconstruction

In many scenarios we want to generate a dense 3D geometry, i.e., a triangle mesh. However, from a multi-view stereo method, or a depth sensor we only obtain an unstructured point cloud. To get a triangle mesh from this unstructured input we need to perform surface reconstruction. In the literature there exists a couple of methods and CloudViewer currently implements the following:

- Alpha shapes [\[Edelsbrunner1983\]](../reference.html#Edelsbrunner1983)
- Ball pivoting [\[Bernardini1999\]](../reference.html#Bernardini1999)
- Poisson surface reconstruction [\[Kazhdan2006\]](../reference.html#Kazhdan2006)

## Alpha shapes
The alpha shape [\[Edelsbrunner1983\]](../reference.html#Edelsbrunner1983) is a generalization of a convex hull. As described [here](https://graphics.stanford.edu/courses/cs268-11-spring/handouts/AlphaShapes/as_fisher.pdf)  one can intuitively
think of an alpha shape as the following: Imagine a huge mass of ice cream containing the points $S$ as hard chocolate pieces. Using one of these sphere-formed ice cream spoons we carve out all parts of the ice cream block we can reach without bumping into chocolate pieces, thereby even carving out holes in the inside (e.g., parts not reachable by simply moving the
spoon from the outside). We will eventually end up with a (not necessarily convex) object bounded by caps, arcs and points. If we now straighten all round faces to triangles and line segments, we have an intuitive description of what is called the alpha shape of $S$.

CloudViewer implements the method `create_from_point_cloud_alpha_shape` that involves the tradeoff parameter `alpha`.

In [2]:
bunny = cv3d.data.BunnyMesh()
mesh = cv3d.io.read_triangle_mesh(bunny.path)
mesh.compute_vertex_normals()

pcd = mesh.sample_points_poisson_disk(750)
cv3d.visualization.draw_geometries([pcd])
alpha = 0.03
print(f"alpha={alpha:.3f}")
mesh = cv3d.geometry.ccMesh.create_from_point_cloud_alpha_shape(pcd, alpha)
mesh.compute_vertex_normals()
cv3d.visualization.draw_geometries([mesh], mesh_show_back_face=True)

[CloudViewer INFO] Downloading https://github.com/isl-org/open3d_downloads/releases/download/20220201-data/BunnyMesh.ply
[CloudViewer INFO] Downloaded to /root/cloudViewer_data/download/BunnyMesh/BunnyMesh.ply
[CloudViewer WARNING] Destination is a directory: /root/cloudViewer_data/extract/BunnyMesh


[CloudViewer WARNING] GLFW Error: X11: The DISPLAY environment variable is missing
[CloudViewer WARNING] GLFW initialized for headless rendering.
[CloudViewer WARNING] Failed to initialize GLEW: Missing GL version (1)


AttributeError: 'NoneType' object has no attribute 'point_show_normal'

The implementation is based on the convex hull of the point cloud. If we want to compute multiple alpha shapes from a given point cloud, then we can save some computation by only computing the convex hull once and pass it to `create_from_point_cloud_alpha_shape`.

In [ ]:
tetra_mesh, pt_map = cv3d.geometry.TetraMesh.create_from_point_cloud(pcd)
for alpha in np.logspace(np.log10(0.5), np.log10(0.01), num=4):
    print(f"alpha={alpha:.3f}")
    mesh = cv3d.geometry.ccMesh.create_from_point_cloud_alpha_shape(
        pcd, alpha, tetra_mesh, pt_map)
    mesh.compute_vertex_normals()
    cv3d.visualization.draw_geometries([mesh], mesh_show_back_face=True)

## Ball pivoting
The ball pivoting algorithm (BPA) [\[Bernardini1999\]](../reference.html#Bernardini1999) is a surface reconstruction method which is related to alpha shapes. Intuitively, think of a 3D ball with a given radius that we drop on the point cloud. If it hits any 3 points (and it does not fall through those 3 points) it creates a triangles. Then, the algorithm starts pivoting from the edges of the existing triangles and every time it hits 3 points where the ball does not fall through we create another triangle.

CloudViewer implements this method in `create_from_point_cloud_ball_pivoting`. The method accepts a list of `radii` as parameter that corresponds to the radii of the individual balls that are pivoted on the point cloud.


<div class="alert alert-info">
    
**Note:** 

This algorithm assumes that the `PointCloud` has `normals`.

</div>

In [ ]:
bunny = cv3d.data.BunnyMesh()
gt_mesh = cv3d.io.read_triangle_mesh(bunny.path)
gt_mesh.compute_vertex_normals()

pcd = gt_mesh.sample_points_poisson_disk(3000)
cv3d.visualization.draw_geometries([pcd])

In [ ]:
radii = [0.005, 0.01, 0.02, 0.04]
rec_mesh = cv3d.geometry.ccMesh.create_from_point_cloud_ball_pivoting(
    pcd, cv3d.utility.DoubleVector(radii))
cv3d.visualization.draw_geometries([pcd, rec_mesh])

## Poisson surface reconstruction
The Poisson surface reconstruction method [\[Kazhdan2006\]](../reference.html#Kazhdan2006) solves a regularized optimization problem to obtain a smooth surface. For this reason, Poisson surface reconstruction can be preferable to the methods mentioned above, as they produce non-smooth results since the points of the `PointCloud` are also the `vertices` of the resulting triangle mesh without any modifications.

CloudViewer implements the method `create_from_point_cloud_poisson` which is basically a wrapper of the code of [Kazhdan](https://github.com/mkazhdan/PoissonRecon). An important parameter of the function is `depth` that defines the depth of the octree used for the surface reconstruction and hence implies the resolution of the resulting triangle mesh. A higher `depth` value means a mesh with more details.

<div class="alert alert-info">
    
**Note:** 

This algorithm assumes that the `PointCloud` has `normals`.

</div>

In [ ]:
eagle = cv3d.data.EaglePointCloud()
pcd = cv3d.io.read_point_cloud(eagle.path)
print(pcd)

cv3d.visualization.draw_geometries([pcd],
                                   zoom=0.664,
                                   front=[-0.4761, -0.4698, -0.7434],
                                   lookat=[1.8900, 3.2596, 0.9284],
                                   up=[0.2304, -0.8825, 0.4101])

In [ ]:
print('run Poisson surface reconstruction')
with cv3d.utility.VerbosityContextManager(
        cv3d.utility.VerbosityLevel.Debug) as cm:
    mesh, densities = cv3d.geometry.ccMesh.create_from_point_cloud_poisson(
        pcd, depth=9)
print(mesh)
cv3d.visualization.draw_geometries([mesh],
                                   zoom=0.664,
                                   front=[-0.4761, -0.4698, -0.7434],
                                   lookat=[1.8900, 3.2596, 0.9284],
                                   up=[0.2304, -0.8825, 0.4101])

Poisson surface reconstruction will also create triangles in areas of low point density, and even extrapolates into some areas (see bottom of the eagle output above). The `create_from_point_cloud_poisson` function has a second `densities` return value that indicates for each vertex the density. A low density value means that the vertex is only supported by a low number of points from the input point cloud.

In the code below we visualize the density in 3D using pseudo color. Violet indicates low density and yellow indicates a high density.

In [ ]:
print('visualize densities')
densities = np.asarray(densities)
density_colors = plt.get_cmap('plasma')(
    (densities - densities.min()) / (densities.max() - densities.min()))
density_colors = density_colors[:, :3]
density_mesh = cv3d.geometry.ccMesh()
density_mesh.create_internal_cloud()
density_mesh.set_vertices(mesh.get_vertices())
density_mesh.set_triangles(mesh.get_triangles())
density_mesh.set_vertex_colors(cv3d.utility.Vector3dVector(density_colors))
cv3d.visualization.draw_geometries([density_mesh],
                                   zoom=0.664,
                                   front=[-0.4761, -0.4698, -0.7434],
                                   lookat=[1.8900, 3.2596, 0.9284],
                                   up=[0.2304, -0.8825, 0.4101])

We can further use the density values to remove vertices and triangles that have a low support. In the code below we remove all vertices (and connected triangles) that have a lower density value than the $0.01$ quantile of all density values.

In [ ]:
print('remove low density vertices')
vertices_to_remove = densities < np.quantile(densities, 0.01)
mesh.remove_vertices_by_mask(vertices_to_remove)
print(mesh)
cv3d.visualization.draw_geometries([mesh],
                                   zoom=0.664,
                                   front=[-0.4761, -0.4698, -0.7434],
                                   lookat=[1.8900, 3.2596, 0.9284],
                                   up=[0.2304, -0.8825, 0.4101])

## Normal estimation
In the examples above we assumed that the point cloud has normals that point outwards. However, not all point clouds already come with associated normals. CloudViewer can be used to estimate point cloud normals with `estimate_normals`, which locally fits a plane per 3D point to derive the normal. However, the estimated normals might not be consistently oriented. `orient_normals_consistent_tangent_plane` propagates the normal orientation using a minimum spanning tree.

In [ ]:
bunny = cv3d.data.BunnyMesh()
gt_mesh = cv3d.io.read_triangle_mesh(bunny.path)

pcd = gt_mesh.sample_points_poisson_disk(5000)
pcd.set_normals(cv3d.utility.Vector3dVector(np.zeros(
    (5000, 3))))  # invalidate existing normals

pcd.estimate_normals()
cv3d.visualization.draw_geometries([pcd], point_show_normal=True)

In [ ]:
pcd.orient_normals_consistent_tangent_plane(100)
cv3d.visualization.draw_geometries([pcd], point_show_normal=True)